# MM4250: stuck check and thru check

**Stuck check**: ALL_OPEN, internal short, ALL_OPEN, internal load, ALL_OPEN, then each port in the cooldown file with ALL_OPEN after it. Every position is compared with its neighbours, so a channel that won't close (stuck open) or won't open (stuck closed) stands out. Run it on the way down in the 30-80 K range and first thing at any temperature where you'll take data.

**Thru check**: the direct test through the fridge's thru line (the file's `lines.thru`, wired to `thru_channel`); it asks you to move VNA port 1's cable and back.

- Only one notebook can hold the switch: run the other notebook's **Close** cell first.
- Nothing is saved unless you ask (`SAVE_DB`: the lab's `..._stuck` database; `SAVE_TOUCHSTONE`: files + `positions.csv` beside it). The switch is always left `ALL_OPEN`, even if a check fails.
- **If anything is stuck, stop**: don't keep switching it cold. Run Close and tell Charlie.

**Setup**: which cooldown file.

In [ ]:
COOLDOWN_FILE = "cooldowns/CD05.json"   # this cooldown's file (see README.md)

import sys
from pathlib import Path
try:
    here = Path(_dh[0])                      # this notebook's folder (Jupyter)
except NameError:
    here = Path.cwd()
root = here
while not (root / "drivers").is_dir() and root != root.parent:
    root = root.parent                       # the folder with drivers/ in it
sys.path[:0] = [str(here), str(root)]
%matplotlib inline

import vna, db, ecal, plots
cfg = db.load_cooldown(here / COOLDOWN_FILE)
db.print_cooldown(cfg)

**Connect**: VNA, switch, fridge (optional: without the VPN it warns and carries on).

In [ ]:
from qcodes.instrument import Instrument
from drivers.KeysightVNA_driver import KeysightP5004B
from drivers.MM4250_QCodes_driver import MM4250

Instrument.close_all()                       # so re-running this cell is safe
ksvna = KeysightP5004B("ksvna", "TCPIP0::QTSF-Measurement::hislip_PXI0_CHASSIS1_SLOT1_INDEX0::INSTR")
switch = MM4250("switch")                    # connecting sets ALL_OPEN

fridge = None                                # only for temperatures: optional
try:
    from maybell_qcodes import MaybellFridge
    fridge = MaybellFridge("fridge", host="10.179.200.168")   # needs the GlobalConnect VPN
except Exception as e:
    print(f"[warning] no fridge ({type(e).__name__}: {e}); carrying on without temperatures")
print("Fridge (mixing plate):", vna.temp_label(vna.read_temperature(fridge)))

**Stuck check**: about a minute. VNA port 1 on the input line, port 2 on the return line, LNA on.

Reading it: each row's *contrast* says how far it stands from what it should differ from (a closed position from the ALL_OPEN before it; an ALL_OPEN from the position before it). Every contrast should be well above 0 dB. A closed position near 0 dB **didn't close** (stuck open); an ALL_OPEN near 0 dB **didn't open** after the position before it (stuck closed). Once one is stuck, later rows can look wrong too: the first one counts. With `stuck_check.pass_db` set in the cooldown file, rows below it are flagged; with `null` (no pass mark yet), nothing is flagged and you read the numbers. Bottom plot: median |S21 - first ALL_OPEN| per position (the old A1 bars; compare neighbours, the chain drifts while cooling).

In [ ]:
SAVE_DB = False          # True: every position into the "stuck" database
SAVE_TOUCHSTONE = False  # True: one .s2p per position + positions.csv in Sweeps/stuck_check/ beside it

STUCK_DB = db.open_database(cfg, "stuck", create=SAVE_DB)
save = db.Saver(STUCK_DB if SAVE_DB else None, "stuck_check", files=SAVE_TOUCHSTONE,
                files_root=STUCK_DB.parent)
stuck = vna.stuck_check(cfg, ksvna, switch, fridge, save=save)      # ends ALL_OPEN
vna.print_check(stuck, cfg["stuck_check"]["pass_db"])
if SAVE_TOUCHSTONE:
    db.write_check_csv(stuck, save.folder)
if SAVE_DB:
    db.backup_db(STUCK_DB)
plots.plot_check(stuck, cfg["switch_serial"]);

**Thru check**: asks you to move VNA port 1's cable from the input line to the thru line (Enter; `n` stops before anything moves), closes and opens `thru_channel` twice, then asks you to move the cable back. S21 is then a transmission: high with the channel closed, down at the switch's isolation with it open. Compared with `stuck_check.thru_pass_db` (20 dB is a guess; the warm run shows the real levels). **Move the cable back** when it asks: takes and the stuck check need port 1 on the input line.

In [ ]:
SAVE_DB = False          # True: every position into the "stuck" database
SAVE_TOUCHSTONE = False  # True: one .s2p per position + positions.csv in Sweeps/thru_check/ beside it

STUCK_DB = db.open_database(cfg, "stuck", create=SAVE_DB)
save = db.Saver(STUCK_DB if SAVE_DB else None, "thru_check", files=SAVE_TOUCHSTONE,
                files_root=STUCK_DB.parent)
thru = vna.thru_check(cfg, ksvna, switch, fridge, save=save)        # asks about the cable; ends ALL_OPEN
if thru is not None:
    vna.print_check(thru)
    if SAVE_TOUCHSTONE:
        db.write_check_csv(thru, save.folder)
    if SAVE_DB:
        db.backup_db(STUCK_DB)
    plots.plot_check(thru, cfg["switch_serial"]);

**Close**: before anyone else needs the switch or VNA. Backs up the stuck database if there is one, leaves the switch `ALL_OPEN` and releases everything.

In [ ]:
STUCK_DB = db.open_database(cfg, "stuck", create=False)
if STUCK_DB.is_file():
    db.backup_db(STUCK_DB)
vna.leave_switch_open(ksvna, switch)          # source off for the move
print("switch, last sent:", switch.state.cache.get(get_if_invalid=False))   # not read back: the lab driver's readback lags
ksvna.output(False)
ksvna.close()
switch.close()
if fridge is not None:
    fridge.close()